# Test the currently saved models

This notebook reports actual **held-out internal participant** predictions for the completed models, with MAE, RMSE, NRMSE, baseline comparisons and measured-versus-predicted waveforms.

Default: **AP_Left / absolute coordinates / SUB01**, using the experiment `reviewer_complete_v1`. The latest training log indicates the four baselines, LSTM and GRU completed this participant's evaluation. Conv1D-LSTM stopped during tuning and is expected to be unavailable until a refitted model exists.

The training notebook already saved `predictions.npz` and `cycles.csv` after each model's outer test. The default mode reads and verifies those predictions. **No training, tuning or GPU is needed for this mode.** Optional modes perform inference from saved weights without fitting anything.

Open this notebook in a separate Colab session, select a CPU runtime, and use **Runtime → Run all**. Mount the same Google Drive account. Outputs are saved in a separate `saved_model_tests` folder. The training experiment is read-only.

The external cohort is not evaluated here. Final external testing still requires the frozen full-cohort model and verified participant, visit, units and coordinate mapping.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, sys, shutil
import numpy as np
import pandas as pd
from IPython.display import display, Image
from google.colab import drive
drive.mount('/content/drive')

PROJECT = Path('/content/drive/MyDrive/Shida_Nested_LOSO')
EXPERIMENT = 'reviewer_complete_v1'
TASK = 'AP_Left'
COORDINATES = 'absolute'
OUTER_PARTICIPANT = 'SUB01'

# cached_only: recommended; read the predictions already computed on SUB01.
# inference_if_needed: use saved weights only when a completed prediction cache is absent.
# fresh_inference: explicitly rerun prediction from each available saved model.
EVALUATION_MODE = 'cached_only'
DATA_ZIP = PROJECT / 'datasets' / 'Shida-LOSO-ANKLE.zip'
STUDY = PROJECT / 'complete_studies' / EXPERIMENT / TASK / COORDINATES
STAMP = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT = PROJECT / 'saved_model_tests' / f'{EXPERIMENT}_{TASK}_{COORDINATES}_{OUTER_PARTICIPANT}_{STAMP}'
CODE = Path('/content/shida_saved_model_evaluation_code')
CODE.mkdir(parents=True,exist_ok=True)
print('Training experiment (read only):',STUDY)
print('Evaluation outputs:',OUTPUT)
if not (STUDY/'prespecification.json').exists():
    raise FileNotFoundError(f'Saved experiment not found: {STUDY}. Check the account and experiment name.')
if not DATA_ZIP.exists():
    raise FileNotFoundError(f'Original internal ZIP not found: {DATA_ZIP}. Use its actual Drive location.')

## Load the evaluation code

The data-loading and preprocessing modules below are copied unchanged from the training notebook. Their hashes are checked against the saved experiment before evaluation. The additional evaluator writes only to the new output folder.

In [ ]:
SOURCES = {'data.py': '"""Read the paired per-participant Raw workbooks in Shida-LOSO-ANKLE."""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom hashlib import sha256\nfrom io import BytesIO\nfrom pathlib import Path\nimport re\nfrom zipfile import ZipFile\n\nimport numpy as np\nimport pandas as pd\n\n\nTASKS = {\n    "apleft": "AP_Left", "apright": "AP_Right",\n    "longlef": "Vertical_Left", "longleft": "Vertical_Left",\n    "longrig": "Vertical_Right", "longright": "Vertical_Right",\n    "verticalleft": "Vertical_Left", "verticalright": "Vertical_Right",\n    "mlleft": "ML_Left", "mlright": "ML_Right",\n}\nTASK_ORDER = ["AP_Left", "AP_Right", "Vertical_Left", "Vertical_Right", "ML_Left", "ML_Right"]\nRAW_PATTERN = re.compile(r"(?:^|/)Raw/([^/]+)/([^/]+\\.xlsx)$", re.I)\nSUBJECT_PATTERN = re.compile(r"(?:SUB|UB)(\\d+)", re.I)\n\n\n@dataclass\nclass Dataset:\n    task: str\n    x: np.ndarray                       # [cycles, 100 time points, 1 feature]\n    y: np.ndarray\n    groups: np.ndarray                  # participant ID for EVERY cycle\n    cycles: pd.DataFrame                # source and cycle-position provenance\n    fingerprint: str\n\n    @property\n    def subjects(self):\n        return sorted(np.unique(self.groups).tolist())\n\n    def subset(self, subjects):\n        mask = np.isin(self.groups, subjects)\n        if not mask.any():\n            raise ValueError("Empty participant subset")\n        fingerprint = sha256((self.fingerprint + repr(sorted(subjects))).encode()).hexdigest()\n        return Dataset(self.task, self.x[mask], self.y[mask], self.groups[mask],\n                       self.cycles.loc[mask].reset_index(drop=True), fingerprint)\n\n\ndef canonical_subject(text):\n    matches = SUBJECT_PATTERN.findall(str(text))\n    ids = {f"SUB{int(value):02d}" for value in matches}\n    if len(ids) != 1:\n        raise ValueError(f"Expected one unambiguous participant in {text!r}; got {ids}")\n    return ids.pop()\n\n\ndef workbook_arrays(payload, filename, subject, expected_timesteps=100):\n    """Keep the original in/out column order. GRF headers lack trial identifiers."""\n    with pd.ExcelFile(BytesIO(payload), engine="openpyxl") as book:\n        if not {"in", "out"}.issubset(book.sheet_names):\n            raise ValueError(f"{filename}: paired \'in\' and \'out\' sheets are required")\n        inputs = pd.read_excel(book, sheet_name="in", header=0)\n        targets = pd.read_excel(book, sheet_name="out", header=0)\n    if inputs.shape != targets.shape or inputs.shape[0] != expected_timesteps or not inputs.shape[1]:\n        raise ValueError(f"{filename}: expected matched ({expected_timesteps}, cycles) sheets; "\n                         f"got in={inputs.shape}, out={targets.shape}")\n    for header in inputs.columns:\n        if canonical_subject(header) != subject:\n            raise ValueError(f"{filename}: input column belongs to a different participant: {header}")\n    try:\n        x = inputs.to_numpy(dtype=np.float32).T[:, :, None]\n        y = targets.to_numpy(dtype=np.float32).T[:, :, None]\n    except (ValueError, TypeError) as exc:\n        raise ValueError(f"{filename}: nonnumeric waveform value") from exc\n    if not np.isfinite(x).all() or not np.isfinite(y).all():\n        raise ValueError(f"{filename}: missing/nonfinite values; no silent zero filling or exclusion is allowed")\n    metadata = [{"subject": subject, "cycle_id": f"{subject}:C{i+1:03d}",\n                 "column_position_1based": i + 1, "input_header": str(inputs.columns[i]),\n                 "target_header": str(targets.columns[i]), "source_workbook": filename}\n                for i in range(len(x))]\n    return x, y, metadata\n\n\ndef load_datasets(source, requested_tasks=None):\n    """Accept the uploaded ZIP, its extracted root, or the Raw directory itself.\n\n    Never load the preassembled Train files: they repeat subjects across folds\n    and some supplied input matrices were concatenated along the wrong axis.\n    """\n    source = Path(source).expanduser().resolve()\n    requested = set(requested_tasks or TASK_ORDER)\n    unknown = requested - set(TASK_ORDER)\n    if unknown:\n        raise ValueError(f"Unknown tasks: {sorted(unknown)}")\n    archive = None\n    if source.is_file() and source.suffix.lower() == ".zip":\n        archive = ZipFile(source)\n        names = sorted(n for n in archive.namelist() if RAW_PATTERN.search(n))\n        reader = archive.read\n    elif source.is_dir():\n        paths = {p.as_posix(): p for p in source.rglob("*.xlsx") if RAW_PATTERN.search(p.as_posix())}\n        names = sorted(paths)\n        reader = lambda name: paths[name].read_bytes()\n    else:\n        raise ValueError(f"Expected an existing ZIP or directory: {source}")\n    accum, seen, fingerprints = {}, set(), {}\n    try:\n        for name in names:\n            match = RAW_PATTERN.search(name)\n            task = TASKS.get(re.sub(r"[^a-z]", "", match.group(1).lower()))\n            if task not in requested:\n                continue\n            subject = canonical_subject(match.group(2))\n            if (task, subject) in seen:\n                raise ValueError(f"Duplicate Raw workbook for {task}/{subject}; provide one dataset root")\n            seen.add((task, subject))\n            payload = reader(name)\n            relative_name = "Raw/" + name.split("/")[-2] + "/" + name.split("/")[-1]\n            x, y, metadata = workbook_arrays(payload, relative_name, subject)\n            accum.setdefault(task, []).append((subject, x, y, metadata))\n            fingerprints.setdefault(task, sha256()).update(relative_name.encode() + sha256(payload).digest())\n    finally:\n        if archive is not None:\n            archive.close()\n    absent = requested - set(accum)\n    if absent:\n        raise ValueError(f"No Raw workbooks found for {sorted(absent)} in {source}")\n    result = {}\n    for task in TASK_ORDER:\n        if task not in accum:\n            continue\n        records = sorted(accum[task], key=lambda item: item[0])\n        if len(records) < 3:\n            raise ValueError(f"{task}: at least three participants are required for nested LOSO")\n        result[task] = Dataset(\n            task, np.concatenate([r[1] for r in records]), np.concatenate([r[2] for r in records]),\n            np.concatenate([np.repeat(r[0], len(r[1])) for r in records]),\n            pd.DataFrame([m for r in records for m in r[3]]), fingerprints[task].hexdigest())\n    return result\n\n\ndef nested_indices(groups, outer_subject):\n    """Yield train, validation, test indices; one entire subject in each holdout."""\n    groups = np.asarray(groups)\n    subjects = sorted(np.unique(groups).tolist())\n    if outer_subject not in subjects or len(subjects) < 3:\n        raise ValueError("Unknown outer participant or fewer than three participants")\n    test = np.flatnonzero(groups == outer_subject)\n    for validation_subject in subjects:\n        if validation_subject == outer_subject:\n            continue\n        validation = np.flatnonzero(groups == validation_subject)\n        train = np.flatnonzero((groups != outer_subject) & (groups != validation_subject))\n        assert_subject_separation(groups, train, validation, test)\n        yield validation_subject, train, validation, test\n\n\ndef assert_subject_separation(groups, train, validation, test):\n    combined = np.concatenate([train, validation, test])\n    if not np.array_equal(np.sort(combined), np.arange(len(groups))):\n        raise ValueError("Each cycle must appear exactly once across the three partitions")\n    sets = [set(np.asarray(groups)[idx]) for idx in [train, validation, test]]\n    if any(not s for s in sets) or sets[0] & sets[1] or sets[0] & sets[2] or sets[1] & sets[2]:\n        raise ValueError("Participant overlap or an empty train/validation/test partition")\n\n\ndef audit_datasets(datasets):\n    tasks, participants, cycles, splits = [], [], [], []\n    for task, ds in datasets.items():\n        tasks.append({"task": task, "n_participants": len(ds.subjects), "n_cycles": len(ds.x),\n                      "timepoints": ds.x.shape[1], "features": ds.x.shape[2],\n                      "missing_input_values": int((~np.isfinite(ds.x)).sum()),\n                      "missing_target_values": int((~np.isfinite(ds.y)).sum()),\n                      "subjects": ds.subjects, "fingerprint": ds.fingerprint})\n        cycles.extend(ds.cycles.assign(task=task).to_dict("records"))\n        for subject in ds.subjects:\n            participants.append({"task": task, "subject": subject,\n                                 "n_cycles": int(np.sum(ds.groups == subject))})\n            for val_subject, tr, va, te in nested_indices(ds.groups, subject):\n                splits.append({"task": task, "outer_test_subject": subject,\n                               "inner_validation_subject": val_subject,\n                               "inner_training_subjects": sorted(set(ds.groups[tr])),\n                               "n_train_cycles": len(tr), "n_validation_cycles": len(va),\n                               "n_test_cycles": len(te)})\n    return {"tasks": tasks, "participants": participants, "cycles": cycles, "splits": splits}\n', 'protocol.py': '"""Backend-independent nested LOSO protocol, metrics, and resumable records."""\nfrom __future__ import annotations\n\nfrom hashlib import sha256\nimport json\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\n\nfrom data import Dataset, nested_indices\n\n\ndef write_json(path, obj):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    temporary.write_text(json.dumps(obj, indent=2, sort_keys=True, allow_nan=False), encoding="utf-8")\n    temporary.replace(path)\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding="utf-8"))\n\n\ndef stable_seed(seed, *labels):\n    text = "|".join(map(str, (seed, *labels)))\n    return int.from_bytes(sha256(text.encode()).digest()[:4], "big") % (2**31 - 1)\n\n\ndef parameter_hash(parameters):\n    return sha256(json.dumps(parameters, sort_keys=True).encode()).hexdigest()\n\n\ndef scale_from_training(train, *others, enabled=False):\n    """Only training inputs determine scale. Targets retain their original units."""\n    if enabled:\n        mean = np.mean(train, axis=(0, 1), keepdims=True, dtype=np.float64)\n        std = np.std(train, axis=(0, 1), keepdims=True, dtype=np.float64)\n        std = np.where(std < 1e-8, 1.0, std)\n    else:\n        mean = np.zeros((1, 1, train.shape[-1]))\n        std = np.ones_like(mean)\n    arrays = [((arr - mean) / std).astype(np.float32) for arr in (train,) + others]\n    return arrays, {"enabled": enabled, "mean": mean.reshape(-1).tolist(),\n                    "std": std.reshape(-1).tolist()}\n\n\ndef cycle_metrics(truth, prediction):\n    if truth.shape != prediction.shape or not np.isfinite(prediction).all():\n        raise ValueError("Invalid prediction shape or nonfinite predictions")\n    error = truth.astype(np.float64) - prediction.astype(np.float64)\n    rmse = np.sqrt(np.mean(error**2, axis=(1, 2)))\n    mae = np.mean(np.abs(error), axis=(1, 2))\n    span = np.ptp(truth.astype(np.float64), axis=1).reshape(-1)\n    nrmse = np.divide(rmse, span, out=np.full_like(rmse, np.nan), where=span > 1e-12)\n    return pd.DataFrame({"rmse": rmse, "nrmse": nrmse, "mae": mae,\n                         "target_range": span, "nrmse_defined": span > 1e-12})\n\n\ndef evaluate_candidate(ds, outer_subject, parameters, backend, seed, directory, scale_input=False):\n    """Score one candidate equally across inner validation participants.\n\n    The backend only receives inner train/validation data. Outer test arrays\n    are never supplied to tuning, early stopping, or input scaling.\n    """\n    directory = Path(directory)\n    signature = parameter_hash(parameters)\n    records = []\n    for validation_subject, tr, va, _ in nested_indices(ds.groups, outer_subject):\n        path = directory / f"inner_{validation_subject}.json"\n        if path.exists():\n            record = read_json(path)\n            if record["parameter_hash"] != signature:\n                raise ValueError(f"Cached trial has different parameters: {path}")\n            records.append(record)\n            continue\n        (xtr, xva), scaler = scale_from_training(ds.x[tr], ds.x[va], enabled=scale_input)\n        fit_seed = stable_seed(seed, ds.task, outer_subject, validation_subject, "inner")\n        fit = backend.fit_inner(xtr, ds.y[tr], xva, ds.y[va], parameters, fit_seed,\n                                directory / f"history_{validation_subject}.csv")\n        metric = cycle_metrics(ds.y[va], fit["prediction"])\n        record = {"outer_subject": outer_subject, "validation_subject": validation_subject,\n                  "training_subjects": sorted(set(ds.groups[tr])), "n_training_cycles": len(tr),\n                  "n_validation_cycles": len(va), "seed": fit_seed,\n                  "parameter_hash": signature, "scaler": scaler,\n                  "best_epoch": int(fit["best_epoch"]), "epochs_run": int(fit["epochs_run"]),\n                  "validation_mae": float(metric.mae.mean()),\n                  "validation_rmse": float(metric.rmse.mean())}\n        write_json(path, record)\n        records.append(record)\n    score = float(np.mean([r["validation_mae"] for r in records]))\n    if not np.isfinite(score):\n        raise ValueError("Nonfinite inner objective")\n    summary = {"parameters": parameters, "inner_subject_mae": score, "folds": records}\n    write_json(directory / "inner_summary.json", summary)\n    return summary\n\n\ndef refit_and_evaluate(ds, outer_subject, parameters, inner_summary, backend, seed,\n                       directory, architecture, scale_input=False):\n    """Refit on ALL outer-training participants for an inner-derived epoch budget."""\n    directory = Path(directory)\n    directory.mkdir(parents=True, exist_ok=True)\n    tr = np.flatnonzero(ds.groups != outer_subject)\n    te = np.flatnonzero(ds.groups == outer_subject)\n    if len(te) == 0 or set(ds.groups[tr]) & set(ds.groups[te]):\n        raise ValueError("Invalid outer subject separation")\n    if set(r["validation_subject"] for r in inner_summary["folds"]) != set(ds.groups[tr]):\n        raise ValueError("The winning trial did not validate every outer-training participant")\n    if any(r["outer_subject"] != outer_subject for r in inner_summary["folds"]):\n        raise ValueError("Inner summary belongs to a different outer fold")\n    if parameter_hash(inner_summary["parameters"]) != parameter_hash(parameters):\n        raise ValueError("Refit parameters differ from the selected inner trial")\n    # Predefined rule: ceiling of median inner best epoch; outer test never selects epochs.\n    epochs = max(1, int(np.ceil(np.median([r["best_epoch"] for r in inner_summary["folds"]]))))\n    (xtr, xte), scaler = scale_from_training(ds.x[tr], ds.x[te], enabled=scale_input)\n    refit_seed = stable_seed(seed, ds.task, outer_subject, "refit")\n    model = backend.fit_final(xtr, ds.y[tr], parameters, refit_seed, epochs,\n                              directory / "final_history.csv")\n    # This is the first point where the fitted outer model sees test inputs.\n    prediction = backend.predict(model, xte)\n    metrics = cycle_metrics(ds.y[te], prediction)\n    metadata = ds.cycles.iloc[te].reset_index(drop=True)\n    per_cycle = pd.concat([metadata, metrics], axis=1)\n    per_cycle.insert(0, "task", ds.task)\n    per_cycle.insert(1, "architecture", architecture)\n    per_cycle.insert(2, "seed", seed)\n    per_cycle.to_csv(directory / "cycle_metrics.csv", index=False)\n    np.savez_compressed(directory / "predictions.npz", measured=ds.y[te], predicted=prediction,\n                        cycle_ids=metadata.cycle_id.to_numpy(dtype=str),\n                        subject_ids=ds.groups[te], gait_phase=np.linspace(0, 100, ds.y.shape[1]))\n    backend.save(model, directory / "model.keras")\n    backend.clear()\n    summary = {"task": ds.task, "architecture": architecture, "seed": seed,\n               "subject": outer_subject, "n_cycles": len(te), "n_train_cycles": len(tr),\n               "n_train_subjects": len(set(ds.groups[tr])), "epochs": epochs,\n               "refit_seed": refit_seed, "rmse": float(metrics.rmse.mean()),\n               "nrmse": float(metrics.nrmse.mean()) if metrics.nrmse.notna().any() else None,\n               "mae": float(metrics.mae.mean()),\n               "undefined_nrmse_cycles": int((~metrics.nrmse_defined).sum()),\n               "inner_subject_mae": inner_summary["inner_subject_mae"],\n               "parameters": parameters, "input_scaler": scaler,\n               "epoch_rule": "ceiling of median best epoch across winning inner LOSO fits"}\n    # Written last: a completed outer fold can be safely skipped on resume.\n    write_json(directory / "result.json", summary)\n    return summary\n\n\ndef summarize_subject_results(records, directory, bootstrap_seed=2026, resamples=2000):\n    """Bootstrap participants; repeated seeds are averaged within participant first."""\n    directory = Path(directory)\n    df = pd.DataFrame([{k: v for k, v in r.items() if not isinstance(v, (list, dict))}\n                       for r in records])\n    if df.empty:\n        return\n    df.to_csv(directory / "outer_subject_metrics_by_seed.csv", index=False)\n    averaged = df.groupby(["task", "architecture", "subject"], as_index=False).agg(\n        rmse=("rmse", "mean"), nrmse=("nrmse", "mean"), mae=("mae", "mean"),\n        n_seeds=("seed", "nunique"), rmse_seed_sd=("rmse", "std"),\n        nrmse_seed_sd=("nrmse", "std"), mae_seed_sd=("mae", "std"))\n    averaged.to_csv(directory / "outer_subject_metrics_seed_average.csv", index=False)\n    rows = []\n    for (task, arch), group in averaged.groupby(["task", "architecture"]):\n        for metric in ["rmse", "nrmse", "mae"]:\n            values = group[metric].dropna().to_numpy()\n            if not len(values):\n                continue\n            rng = np.random.default_rng(stable_seed(bootstrap_seed, task, arch, metric))\n            if len(values) >= 2:\n                boot = np.median(rng.choice(values, size=(resamples, len(values)), replace=True), axis=1)\n                low, high = np.quantile(boot, [0.025, 0.975])\n            else:\n                low = high = np.nan\n            rows.append({"task": task, "architecture": arch, "metric": metric,\n                         "n_participants": len(values), "min_seeds_per_subject": int(group.n_seeds.min()),\n                         "max_seeds_per_subject": int(group.n_seeds.max()),\n                         "median": float(np.median(values)), "q25": float(np.quantile(values, .25)),\n                         "q75": float(np.quantile(values, .75)),\n                         "median_ci95_lower": low, "median_ci95_upper": high,\n                         "ci_method": "participant percentile bootstrap after averaging seeds"})\n    pd.DataFrame(rows).to_csv(directory / "cohort_summary.csv", index=False)\n', 'revision_data.py': '"""Strict cohort contract and coordinate controls; no guessed anatomical data."""\nfrom dataclasses import dataclass, replace\nfrom hashlib import sha256\nfrom pathlib import Path\nimport json\nimport numpy as np\nimport pandas as pd\nfrom data import Dataset, load_datasets\nfrom protocol import write_json, read_json\n\nMODES = (\'absolute\', \'translation\', \'pelvis\', \'aligned\', \'body\', \'combined\')\nAXIS = {\'AP\': 0, \'ML\': 1, \'Vertical\': 2}\nREQUIRED_CONTEXT = [\'cohort_id\', \'dataset_role\', \'ankle_units\', \'grf_units\',\n                    \'coordinate_definition\', \'motion_capture_system\', \'force_platforms\',\n                    \'kinematics_sampling_hz\', \'force_sampling_hz\', \'walking_protocol\',\n                    \'harmonization_description\', \'cycle_definition\']\n\ndef digest_file(path):\n    h = sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda: f.read(1024 * 1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\ndef bool_column(series):\n    mapping = {\'true\': True, \'1\': True, \'yes\': True, \'false\': False, \'0\': False, \'no\': False}\n    v = series.astype(str).str.lower().str.strip().map(mapping)\n    if v.isna().any():\n        raise ValueError(\'include must contain explicit true/false values\')\n    return v.astype(bool)\n\ndef read_table(path, columns):\n    df = pd.read_csv(path, dtype=str, keep_default_na=False)\n    if set(columns) - set(df):\n        raise ValueError(f\'{path}: missing columns {set(columns) - set(df)}\')\n    for col in set(columns) - {\'exclusion_reason\'}:\n        if df[col].str.strip().eq(\'\').any():\n            raise ValueError(f\'{path}: blank required {col}\')\n    return df\n\n@dataclass\nclass Cohort:\n    ds: Dataset\n    context: dict\n    participants: pd.DataFrame\n    trials: pd.DataFrame\n    ledger: pd.DataFrame\n    arrays: dict\n\ndef legacy_cohort(archive, task):\n    ds = load_datasets(archive, [task])[task]\n    meta = ds.cycles.copy()\n    meta[\'participant_id\'] = meta.subject\n    meta[\'trial_id\'] = \'unknown\'  # Never invent trial IDs from cycle positions.\n    meta[\'visit\'] = \'internal\'\n    meta[\'side\'] = task.split(\'_\')[-1].lower()\n    meta[\'include\'] = True\n    meta[\'exclusion_reason\'] = \'\'\n    ds = replace(ds, cycles=meta)\n    context = {k: \'unknown\' for k in REQUIRED_CONTEXT}\n    context.update(cohort_id=\'Shida_internal\', dataset_role=\'internal\', ankle_units=\'mm\',\n                   grf_units=\'N/kg\', units_verified=False,\n                   coordinate_definition=\'Original single component; physical frame unverified\',\n                   source=\'Paired Raw sheets; manuscript units assumed, not independently verified\')\n    people = pd.DataFrame({\'participant_id\': ds.subjects})\n    return Cohort(ds, context, people, pd.DataFrame(), meta, {})\n\ndef load_cohort(directory, task, visits=None, role=None):\n    """Read one side/component cohort bundle. Excluded cycles need no waveform."""\n    root = Path(directory)\n    context = read_json(root / \'cohort.json\')\n    if any(k not in context or context[k] is None for k in REQUIRED_CONTEXT):\n        raise ValueError(\'Complete cohort.json; use "unknown" explicitly for unavailable descriptive fields\')\n    if role and context[\'dataset_role\'] != role:\n        raise ValueError(\'Wrong dataset_role\')\n    if context[\'ankle_units\'] not in (\'m\', \'mm\') or context[\'grf_units\'] != \'N/kg\':\n        raise ValueError(\'Convert ankle units to m/mm and GRF to N/kg before making the bundle\')\n    if context.get(\'task\') != task:\n        raise ValueError(\'cohort.json task differs from requested component/side\')\n    people = read_table(root / \'participants.csv\', [\'participant_id\'])\n    trials = read_table(root / \'trials.csv\', [\'participant_id\', \'trial_id\', \'visit\', \'side\', \'include\', \'exclusion_reason\'])\n    ledger = read_table(root / \'cycles.csv\', [\'cycle_id\', \'participant_id\', \'trial_id\', \'visit\', \'side\', \'include\', \'exclusion_reason\'])\n    if people.participant_id.duplicated().any() or ledger.cycle_id.duplicated().any():\n        raise ValueError(\'Participant and cycle IDs must be unique within this cohort bundle\')\n    if not people.participant_id.str.fullmatch(r\'[A-Za-z0-9_-]+\').all():\n        raise ValueError(\'Use letters, digits, underscores, or hyphens in participant IDs\')\n    keys = [\'participant_id\', \'trial_id\', \'visit\', \'side\']\n    if trials.duplicated(keys).any():\n        raise ValueError(\'Duplicate trial record\')\n    for frame in (trials, ledger):\n        frame[\'include\'] = bool_column(frame.include)\n        if frame.loc[~frame.include, \'exclusion_reason\'].str.strip().eq(\'\').any():\n            raise ValueError(\'Every excluded trial/cycle requires a reason\')\n        if set(frame.participant_id) - set(people.participant_id):\n            raise ValueError(\'Participant missing from participants.csv\')\n        if not set(frame.visit).issubset({\'baseline\', \'post_intervention\', \'internal\'}):\n            raise ValueError(\'Use baseline, post_intervention, or internal visit labels\')\n        if set(frame.side) != {task.split(\'_\')[-1].lower()}:\n            raise ValueError(\'Each bundle must describe just its selected side\')\n    joined = ledger.merge(trials[keys + [\'include\']], on=keys, how=\'left\', suffixes=(\'\', \'_trial\'), validate=\'many_to_one\')\n    if joined.include_trial.isna().any() or (joined.include & ~joined.include_trial.astype(bool)).any():\n        raise ValueError(\'Every cycle must link to a trial; an excluded trial cannot have an included cycle\')\n    with np.load(root / \'waveforms.npz\', allow_pickle=False) as f:\n        arrays = {k: f[k] for k in f.files}\n    ids = arrays.pop(\'cycle_ids\').astype(str)\n    valid = ledger[ledger.include].copy().set_index(\'cycle_id\', drop=False)\n    if len(set(ids)) != len(ids) or set(ids) != set(valid.index):\n        raise ValueError(\'NPZ cycle_ids must match ALL included cycles, one-to-one, before visit filtering\')\n    meta = valid.loc[ids].reset_index(drop=True)\n    n = len(ids)\n    for k, arr in arrays.items():\n        if len(arr) != n or not np.issubdtype(arr.dtype, np.number) or not np.isfinite(arr).all():\n            raise ValueError(f\'{k}: wrong cycle count or nonfinite/nonnumeric values\')\n    if \'ankle_xyz\' in arrays:\n        xyz = arrays[\'ankle_xyz\']\n        if xyz.shape != (n, 100, 3) or arrays.get(\'grf_xyz\', np.empty(0)).shape != xyz.shape:\n            raise ValueError(\'ankle_xyz and grf_xyz must both be [N,100,3]\')\n        if context.get(\'canonical_axis_order\') != [\'forward\', \'left\', \'up\']:\n            raise ValueError(\'Map both kinematics and GRFs into the same right-handed [forward,left,up] laboratory axes\')\n        if arrays.get(\'direction_xyz\', np.empty(0)).shape != (n, 3):\n            raise ValueError(\'Provide a force-independent horizontal walking direction for every cycle\')\n        basis = gait_basis(arrays[\'direction_xyz\'])\n        axis = AXIS[task.split(\'_\')[0]]\n        x = xyz[:, :, axis:axis + 1]\n        # SAME gait-frame targets for ALL coordinate variants, enabling fair comparisons.\n        y = np.einsum(\'nti,nij->ntj\', arrays[\'grf_xyz\'], basis)[:, :, axis:axis + 1]\n    else:\n        x, y = arrays.get(\'x\'), arrays.get(\'y\')\n        if x is None or y is None or x.shape != (n, 100, 1) or y.shape != x.shape:\n            raise ValueError(\'Scalar bundles require x and y, each [N,100,1]\')\n    take = np.ones(n, bool) if visits is None else meta.visit.isin(visits).to_numpy()\n    if not take.any():\n        raise ValueError(\'No valid cycles for the prespecified visit policy\')\n    meta = meta.loc[take].reset_index(drop=True)\n    meta[\'subject\'] = meta.participant_id\n    fingerprint = sha256(\'\'.join(digest_file(root / f) for f in\n                         [\'cohort.json\', \'participants.csv\', \'trials.csv\', \'cycles.csv\', \'waveforms.npz\']).encode()).hexdigest()\n    fingerprint = sha256((fingerprint + repr(visits)).encode()).hexdigest()\n    ds = Dataset(task, x[take].astype(\'float32\'), y[take].astype(\'float32\'),\n                 meta.participant_id.to_numpy(str), meta, fingerprint)\n    return Cohort(ds, context, people, trials, ledger, {k: v[take] for k, v in arrays.items()})\n\ndef gait_basis(direction):\n    forward = np.array(direction, float, copy=True)\n    if forward.ndim != 2 or forward.shape[1] != 3 or not np.isfinite(forward).all():\n        raise ValueError(\'Invalid direction vectors\')\n    if np.any(np.abs(forward[:, 2]) > 1e-6):\n        raise ValueError(\'Walking directions must be projected onto the horizontal plane\')\n    length = np.linalg.norm(forward, axis=1)\n    if np.any(length < 1e-8):\n        raise ValueError(\'Zero walking direction; supply verified direction from pelvis progression/protocol\')\n    forward /= length[:, None]\n    up = np.tile([0., 0., 1.], (len(forward), 1))\n    left = np.cross(up, forward)\n    return np.stack([forward, left, up], axis=2)\n\ndef coordinate_dataset(cohort, mode=\'absolute\', translation_m=None, rotation_deg=0):\n    if mode not in MODES:\n        raise ValueError(f\'Unknown coordinate mode {mode}\')\n    ds, a = cohort.ds, cohort.arrays\n    units = cohort.context[\'ankle_units\']\n    factor = 1000. if units == \'mm\' else 1.\n    axis = AXIS[ds.task.split(\'_\')[0]]\n    if \'ankle_xyz\' not in a:\n        if mode not in (\'absolute\', \'translation\') or rotation_deg:\n            raise ValueError(\'This analysis requires verified 3D/pelvis/body-scale data; scalar inputs cannot supply it\')\n        x = ds.x.astype(float).copy()\n        if translation_m is not None:\n            if not cohort.context.get(\'units_verified\', False):\n                raise ValueError(\'Verify physical units before metre-based perturbations\')\n            x += np.asarray(translation_m)[axis] * factor\n        if mode == \'translation\':\n            x -= x[:, :1, :]\n    else:\n        x = a[\'ankle_xyz\'].astype(float).copy()\n        pelvis = a.get(\'pelvis_xyz\')\n        pelvis = None if pelvis is None else pelvis.astype(float).copy()\n        direction = a[\'direction_xyz\'].astype(float).copy()\n        if rotation_deg:\n            theta = np.deg2rad(rotation_deg)\n            R = np.array([[np.cos(theta), -np.sin(theta), 0], [np.sin(theta), np.cos(theta), 0], [0, 0, 1]])\n            x = x @ R.T\n            direction = direction @ R.T\n            if pelvis is not None:\n                pelvis = pelvis @ R.T\n        if translation_m is not None:\n            if not cohort.context.get(\'units_verified\', False):\n                raise ValueError(\'Verify physical units before metre-based perturbations\')\n            offset = np.asarray(translation_m, float) * factor\n            if offset.shape != (3,):\n                raise ValueError(\'Translation requires three metre offsets\')\n            x += offset\n            if pelvis is not None:\n                pelvis += offset\n        if mode in (\'pelvis\', \'combined\'):\n            if pelvis is None or pelvis.shape != x.shape:\n                raise ValueError(\'Synchronized pelvis_xyz [N,100,3] is required\')\n            x -= pelvis\n        elif mode in (\'translation\', \'aligned\', \'body\'):\n            x -= x[:, :1, :]\n        if mode in (\'aligned\', \'combined\'):\n            x = np.einsum(\'nti,nij->ntj\', x, gait_basis(direction))\n        if mode in (\'body\', \'combined\'):\n            if not cohort.context.get(\'units_verified\', False) or not cohort.context.get(\'body_scale_definition\'):\n                raise ValueError(\'Verify coordinate units and prespecify height or leg-length definition\')\n            height = a.get(\'body_scale_m\')\n            if height is None or height.shape != (len(x),) or np.any(height <= 0):\n                raise ValueError(\'Positive body_scale_m [N], with a fixed definition, is required\')\n            x /= height[:, None, None] * factor\n        x = x[:, :, axis:axis + 1]\n    fingerprint = sha256((ds.fingerprint + repr((mode, translation_m, rotation_deg))).encode()).hexdigest()\n    return replace(ds, x=x.astype(\'float32\'), fingerprint=fingerprint)\n\ndef write_templates(directory):\n    """Blank data contract, not fabricated external observations."""\n    root = Path(directory)\n    root.mkdir(parents=True, exist_ok=True)\n    for name, columns in {\n        \'participants.csv\': [\'participant_id\', \'global_subject_id\', \'age\', \'sex\', \'medication_state\', \'HY_stage\', \'intervention_group\'],\n        \'trials.csv\': [\'participant_id\', \'trial_id\', \'visit\', \'side\', \'include\', \'exclusion_reason\'],\n        \'cycles.csv\': [\'cycle_id\', \'participant_id\', \'trial_id\', \'visit\', \'side\', \'include\', \'exclusion_reason\'],\n    }.items():\n        if not (root / name).exists():\n            pd.DataFrame(columns=columns).to_csv(root / name, index=False)\n    context = {k: None for k in REQUIRED_CONTEXT}\n    context.update(task=None, units_verified=False, harmonization_verified=False, participant_disjointness_verified=False,\n                   canonical_axis_order=[\'forward\', \'left\', \'up\'], body_scale_definition=None)\n    if not (root / \'cohort.json\').exists():\n        write_json(root / \'cohort.json\', context)\n    return root\n\ndef harmonize_input_units(cohort, training_units):\n    """Deterministic physical-unit conversion, never fitted on external data."""\n    units = cohort.context[\'ankle_units\']\n    if units not in (\'m\', \'mm\') or training_units not in (\'m\', \'mm\'):\n        raise ValueError(\'Known internal and external length units are required\')\n    if units == training_units:\n        return cohort\n    multiplier = 1000. if units == \'m\' else .001\n    arrays = dict(cohort.arrays)\n    for key in [\'ankle_xyz\', \'pelvis_xyz\']:\n        if key in arrays:\n            arrays[key] = arrays[key] * multiplier\n    ds = replace(cohort.ds, x=cohort.ds.x * multiplier)\n    return replace(cohort, ds=ds, arrays=arrays,\n                   context={**cohort.context, \'ankle_units\': training_units})\n\ndef pack_scalar_workbooks(input_xlsx, target_xlsx, cycle_map_csv, participants_csv,\n                          trials_csv, cohort_json, output):\n    """Convert explicitly mapped old-format Excel cycles; never infer participants."""\n    import shutil\n    root = Path(output)\n    root.mkdir(parents=True, exist_ok=True)\n    ledger = read_table(cycle_map_csv, [\'cycle_id\', \'participant_id\', \'trial_id\', \'visit\', \'side\',\n                                       \'include\', \'exclusion_reason\'])\n    if {\'input_column\', \'target_column\'} - set(ledger):\n        raise ValueError(\'Explicit input_column and target_column mappings are required\')\n    accepted = ledger[bool_column(ledger.include)]\n    inputs = pd.read_excel(input_xlsx, engine=\'openpyxl\')\n    targets = pd.read_excel(target_xlsx, engine=\'openpyxl\')\n    if inputs.shape[0] != 100 or targets.shape[0] != 100:\n        raise ValueError(\'Prepare matched complete cycles at 100 normalized gait phases before packing; no padding/resampling is guessed\')\n    xs, ys = [], []\n    for row in accepted.itertuples():\n        x = inputs[row.input_column].to_numpy(dtype=\'float32\')\n        y = targets[row.target_column].to_numpy(dtype=\'float32\')\n        if not np.isfinite(x).all() or not np.isfinite(y).all():\n            raise ValueError(f\'{row.cycle_id}: missing values; document exclusion or fix the source, never zero-fill\')\n        xs.append(x[:, None]); ys.append(y[:, None])\n    if not xs:\n        raise ValueError(\'No included cycles\')\n    np.savez_compressed(root / \'waveforms.npz\', cycle_ids=accepted.cycle_id.to_numpy(str), x=np.stack(xs), y=np.stack(ys))\n    ledger.to_csv(root / \'cycles.csv\', index=False)\n    for source, filename in [(participants_csv, \'participants.csv\'), (trials_csv, \'trials.csv\'), (cohort_json, \'cohort.json\')]:\n        if Path(source).resolve() != (root / filename).resolve():\n            shutil.copy2(source, root / filename)\n    context = read_json(root / \'cohort.json\')\n    load_cohort(root, context[\'task\'], role=context[\'dataset_role\'])\n    return root\n', 'evaluate_saved.py': '"""Read-only held-out evaluation. This module never trains or tunes a model."""\nfrom pathlib import Path\nimport hashlib\nimport importlib.metadata\nimport json\nimport numpy as np\nimport pandas as pd\nfrom protocol import cycle_metrics\n\nFAMILIES = (\'mean_waveform\', \'ridge\', \'phase_only\', \'shallow_mlp\', \'lstm\', \'gru\', \'conv1d_lstm\')\n\ndef read_json(path):\n    return json.loads(Path(path).read_text())\n\ndef digest(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(1024*1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\ndef check_metadata(spec, info, selected, family, outer, ds):\n    if spec[\'config\'].get(\'smoke\', False):\n        raise ValueError(\'This evaluator requires research outputs, not smoke-test fixtures\')\n    if spec[\'task\'] != ds.task or spec[\'data_fingerprint\'] != ds.fingerprint:\n        raise ValueError(\'Internal data or preprocessing differs from the saved experiment\')\n    if sorted(spec[\'subjects\']) != ds.subjects or outer not in ds.subjects:\n        raise ValueError(\'Participant list differs from the saved experiment\')\n    if info[\'family\'] != family or selected[\'family\'] != family:\n        raise ValueError(\'Saved family mismatch\')\n    training = info.get(\'training_participants\', [])\n    expected = [s for s in ds.subjects if s != outer]\n    if outer in training or sorted(training) != expected or len(training) != len(set(training)):\n        raise ValueError(\'Test participant overlaps training, or training cohort is incomplete\')\n    if info.get(\'n_training_participants\') != len(expected):\n        raise ValueError(\'Saved training-participant count differs\')\n    if info.get(\'data_fingerprint\') != ds.subset(expected).fingerprint:\n        raise ValueError(\'Model training-data fingerprint does not match\')\n    if info.get(\'parameters\') != selected.get(\'parameters\'):\n        raise ValueError(\'Saved refit parameters differ from the frozen selection\')\n    if family in (\'lstm\', \'gru\', \'conv1d_lstm\') and info.get(\'epochs\') != selected.get(\'epochs\'):\n        raise ValueError(\'Saved refit epochs differ from the frozen selection\')\n\ndef read_prediction_cache(path, te):\n    with np.load(path, allow_pickle=False) as cache:\n        ids = cache[\'cycle_ids\'].astype(str)\n        truth = cache[\'measured\']\n        pred = cache[\'predicted\']\n    expected = te.cycles.cycle_id.to_numpy(str)\n    if not np.array_equal(ids, expected):\n        raise ValueError(\'Saved predictions have different cycle identities/order\')\n    if truth.shape != te.y.shape or pred.shape != te.y.shape:\n        raise ValueError(\'Saved prediction or target shape differs from test data\')\n    if not np.isfinite(truth).all() or not np.isfinite(pred).all():\n        raise ValueError(\'Nonfinite cached targets or predictions\')\n    if not np.allclose(truth, te.y, rtol=1e-6, atol=1e-7):\n        raise ValueError(\'Cached targets differ from the original held-out measurements\')\n    return pred\n\ndef predict_weights(directory, info, x, software):\n    """Inference only, using the input scaler saved with the fitted model."""\n    path = Path(directory)/info[\'file\']\n    if info[\'family\'] in (\'lstm\',\'gru\',\'conv1d_lstm\'):\n        for name in [\'tensorflow\',\'keras\']:\n            actual = importlib.metadata.version(name)\n            if actual != software.get(name):\n                raise ValueError(f\'{name}: inference needs the saved version {software.get(name)}, found {actual}. Cached predictions do not require TensorFlow.\')\n        import tensorflow as tf\n        scaler = info[\'input_scaler\']\n        std = np.asarray(scaler[\'std\'], float)\n        if not np.isfinite(std).all() or np.any(std <= 0):\n            raise ValueError(\'Invalid saved scaler\')\n        z = (x - np.asarray(scaler[\'mean\'])[None,None,:]) / std[None,None,:]\n        model = tf.keras.models.load_model(path, compile=False)\n        try:\n            return np.asarray(model.predict(z, verbose=0))\n        finally:\n            del model\n            tf.keras.backend.clear_session()\n    for name in [\'scikit-learn\',\'joblib\']:\n        actual = importlib.metadata.version(name)\n        if actual != software.get(name):\n            raise ValueError(f\'{name}: saved-model version is {software.get(name)}, current version is {actual}. Use cached predictions or the original package versions.\')\n    import joblib\n    # Only load trusted model files created by the user\'s own training notebook.\n    obj = joblib.load(path)\n    if obj[\'family\'] != info[\'family\']:\n        raise ValueError(\'Baseline family differs inside saved model\')\n    if \'waveform\' in obj:\n        return np.repeat(obj[\'waveform\'][None], len(x), axis=0)\n    z = obj[\'scaler\'].transform(x.reshape(len(x),-1))\n    return np.asarray(obj[\'model\'].predict(z)).reshape((len(x),*obj[\'shape\']))\n\ndef evaluate_available(root, ds, outer, output, mode=\'cached_only\'):\n    if mode not in (\'cached_only\',\'inference_if_needed\',\'fresh_inference\'):\n        raise ValueError(\'Unknown evaluation mode\')\n    root, output = Path(root), Path(output)\n    if output.resolve().is_relative_to(root.resolve()):\n        raise ValueError(\'Write test reports outside the training experiment\')\n    output.mkdir(parents=True,exist_ok=True)\n    spec = read_json(root/\'prespecification.json\')\n    te = ds.subset([outer])\n    inventory, tables, summary, predictions, hashes = [], [], [], {}, {}\n    for family in FAMILIES:\n        folder = root/\'nested\'/outer/family\n        descriptor, selection = folder/\'refit/model.json\', folder/\'selection/selected.json\'\n        if not descriptor.exists() or not selection.exists():\n            inventory.append(dict(model=family,status=\'unavailable\',reason=\'No completed frozen selection/refitted model\'))\n            continue\n        try:\n            info, chosen = read_json(descriptor), read_json(selection)\n            check_metadata(spec,info,chosen,family,outer,ds)\n            marker, cache = folder/\'complete.json\', folder/\'predictions.npz\'\n            use_cache = mode != \'fresh_inference\' and cache.exists() and marker.exists()\n            if use_cache:\n                done = read_json(marker)\n                if done.get(\'outer_subject\') != outer or done.get(\'family\') != family:\n                    raise ValueError(\'Outer-evaluation completion marker differs\')\n                pred = read_prediction_cache(cache,te)\n                kind = \'saved held-out predictions\'\n                hashes[family] = {\'predictions_sha256\':digest(cache)}\n            else:\n                if mode == \'cached_only\':\n                    inventory.append(dict(model=family,status=\'unavailable\',reason=\'Completed prediction cache absent; inference mode can use a saved refit\'))\n                    continue\n                weights = folder/\'refit\'/info[\'file\']\n                if not weights.is_file():\n                    raise ValueError(\'Refitted weights missing\')\n                pred = predict_weights(folder/\'refit\',info,te.x,spec[\'software\'])\n                kind = \'fresh inference from saved weights\'\n                hashes[family] = {\'model_sha256\':digest(weights)}\n            metrics = cycle_metrics(te.y,pred)\n            a = te.y.reshape(len(te.y),-1).astype(float)\n            b = pred.reshape(len(pred),-1).astype(float)\n            a -= a.mean(axis=1,keepdims=True)\n            b -= b.mean(axis=1,keepdims=True)\n            denominator = np.linalg.norm(a,axis=1)*np.linalg.norm(b,axis=1)\n            metrics[\'waveform_r\'] = np.divide((a*b).sum(axis=1),denominator,\n                out=np.full(len(a),np.nan),where=denominator>1e-12)\n            metrics.insert(0,\'cycle_id\',te.cycles.cycle_id.to_numpy(str))\n            metrics.insert(0,\'participant_id\',outer)\n            metrics.insert(0,\'model\',family)\n            metrics.insert(0,\'task\',te.task)\n            metrics[\'prediction_source\'] = kind\n            tables.append(metrics)\n            summary.append(dict(task=te.task,participant_id=outer,model=family,n_participants=1,\n                n_cycles=len(te.y),mae=float(metrics.mae.mean()),rmse=float(metrics.rmse.mean()),\n                nrmse=float(metrics.nrmse.mean()),nrmse_percent=float(metrics.nrmse.mean()*100),\n                mean_cycle_waveform_r=float(metrics.waveform_r.mean()),\n                undefined_nrmse_cycles=int(metrics.nrmse.isna().sum()),prediction_source=kind,\n                uncertainty=\'No participant-level confidence interval from one participant\'))\n            predictions[family] = pred\n            hashes[family].update(model_metadata_sha256=digest(descriptor),selection_sha256=digest(selection))\n            np.savez_compressed(output/f\'{family}_heldout_predictions.npz\',\n                cycle_ids=te.cycles.cycle_id.to_numpy(str),measured=te.y,predicted=pred)\n            inventory.append(dict(model=family,status=\'evaluated\',reason=kind))\n        except (ValueError,KeyError,OSError,importlib.metadata.PackageNotFoundError) as error:\n            inventory.append(dict(model=family,status=\'not evaluated\',reason=str(error)))\n    inventory = pd.DataFrame(inventory)\n    inventory.to_csv(output/\'model_availability.csv\',index=False)\n    result = pd.DataFrame(summary)\n    if tables:\n        pd.concat(tables,ignore_index=True).to_csv(output/\'heldout_cycle_metrics.csv\',index=False)\n        result.to_csv(output/\'heldout_participant_metrics.csv\',index=False)\n        if \'mean_waveform\' in set(result.model):\n            reference = result.set_index(\'model\').loc[\'mean_waveform\']\n            comparisons=[]\n            for _, row in result.iterrows():\n                if row.model == \'mean_waveform\': continue\n                comparisons.append(dict(model=row.model,reference=\'mean_waveform\',n_participants=1,\n                    rmse_improvement=reference.rmse-row.rmse,\n                    nrmse_percentage_point_improvement=reference.nrmse_percent-row.nrmse_percent,\n                    interpretation=\'Positive favors model; descriptive result for one held-out participant\'))\n            pd.DataFrame(comparisons).to_csv(output/\'descriptive_baseline_comparisons.csv\',index=False)\n    (output/\'evaluation_provenance.json\').write_text(json.dumps({\n        \'experiment\':str(root),\'task\':ds.task,\'held_out_participant\':outer,\n        \'n_test_participants\':1,\'test_cycle_count\':len(te.y),\'mode\':mode,\n        \'prespecification_sha256\':digest(root/\'prespecification.json\'),\'artifacts\':hashes,\n        \'notes\':[\'No training or tuning performed\',\'No external cohort loaded\',\n                 \'No selected_deep winner declared from these test results\',\n                 \'NRMSE = cycle RMSE / measured full-cycle range; participant result = mean of defined cycle values\',\n                 \'Cycles are repeated observations within one participant; no cohort inference or significance tests\']},indent=2))\n    return inventory,result,predictions,te\n\ndef make_plots(summary, predictions, te, output):\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    from matplotlib.ticker import PercentFormatter\n    from math import ceil\n    output=Path(output)\n    plt.rcParams.update({\'font.size\':10,\'axes.spines.top\':False,\'axes.spines.right\':False,\'pdf.fonttype\':42})\n    labels={\'mean_waveform\':\'Mean waveform\',\'ridge\':\'Ridge\',\'phase_only\':\'Phase only\',\n            \'shallow_mlp\':\'Shallow MLP\',\'lstm\':\'LSTM\',\'gru\':\'GRU\',\'conv1d_lstm\':\'Conv1D-LSTM\'}\n    order=[m for m in FAMILIES if m in predictions]\n    if not order: return []\n    data=summary.set_index(\'model\').loc[order]\n    colors=[\'#6C7A89\' if m not in (\'lstm\',\'gru\',\'conv1d_lstm\') else {\'lstm\':\'#0072B2\',\'gru\':\'#D58A00\',\'conv1d_lstm\':\'#009E73\'}[m] for m in order]\n    subtitle=f\'{te.task} | held-out participant {te.subjects[0]} | {len(te.y)} cycles | n = 1 participant\'\n    fig,axes=plt.subplots(1,3,figsize=(15,4.8))\n    for ax,metric,title in zip(axes,[\'mae\',\'rmse\',\'nrmse_percent\'],[\'Mean cycle MAE\',\'Mean cycle RMSE\',\'Mean cycle NRMSE (%)\']):\n        values=data[metric].to_numpy()\n        ax.barh(range(len(order)),values,color=colors)\n        ax.set_yticks(range(len(order)),[labels[m] for m in order]);ax.invert_yaxis()\n        ax.set_xlabel(\'Target units\' if metric!=\'nrmse_percent\' else \'Percent of measured cycle range\')\n        ax.set_title(title);ax.set_xlim(left=0);ax.grid(axis=\'x\',alpha=.15)\n    fig.suptitle(\'Saved-model held-out errors\\n\'+subtitle,fontsize=14)\n    fig.text(.02,.015,\'Descriptive participant-level results. No cohort confidence intervals. Physical target units require source verification.\',fontsize=9)\n    fig.tight_layout(rect=(0,.07,1,.89))\n    fig.savefig(output/\'Heldout_Model_Errors.png\',dpi=180)\n    fig.savefig(output/\'Heldout_Model_Errors.pdf\');plt.close(fig)\n    phase=np.linspace(0,100,te.y.shape[1]);truth=te.y[:,:,0]\n    fig,axes=plt.subplots(ceil(len(order)/3),3,figsize=(15,3.5*ceil(len(order)/3)),squeeze=False,sharex=True,sharey=True)\n    for ax,family,color in zip(axes.flat,order,colors):\n        low,high=np.quantile(truth,[.25,.75],axis=0)\n        ax.fill_between(phase,low,high,color=\'#ADB5BD\',alpha=.25,label=\'Measured cycle IQR\')\n        ax.plot(phase,truth.mean(axis=0),color=\'black\',lw=1.8,label=\'Measured mean\')\n        ax.plot(phase,predictions[family][:,:,0].mean(axis=0),color=color,lw=1.8,label=\'Predicted mean\')\n        ax.set_title(labels[family]);ax.set_xlabel(\'Normalized gait phase (%)\');ax.set_ylabel(\'GRF (target units)\')\n        ax.grid(alpha=.15)\n    for ax in list(axes.flat)[len(order):]:ax.set_visible(False)\n    handles,legend=axes.flat[0].get_legend_handles_labels()\n    fig.legend(handles,legend,loc=\'lower center\',ncol=3,frameon=False)\n    fig.suptitle(\'Measured and predicted waveforms\\n\'+subtitle,fontsize=14)\n    fig.tight_layout(rect=(0,.07,1,.90))\n    fig.savefig(output/\'Heldout_Waveforms.png\',dpi=180)\n    fig.savefig(output/\'Heldout_Waveforms.pdf\');plt.close(fig)\n    k=min(3,len(truth));fig,axes=plt.subplots(k,1,figsize=(11,3*k),squeeze=False,sharex=True)\n    for i,ax in enumerate(axes[:,0]):\n        ax.plot(phase,truth[i],color=\'black\',lw=2.2,label=\'Measured\')\n        for j,(family,color) in enumerate(zip(order,colors)):\n            ax.plot(phase,predictions[family][i,:,0],color=color,ls=[\'--\',\':\',\'-.\',\'--\',\'-\',\'-\',\'-\'][FAMILIES.index(family)],lw=1.4,label=labels[family])\n        ax.set_title(str(te.cycles.cycle_id.iloc[i]));ax.set_ylabel(\'GRF (target units)\');ax.grid(alpha=.15)\n    axes[-1,0].set_xlabel(\'Normalized gait phase (%)\')\n    fig.legend(*axes[0,0].get_legend_handles_labels(),loc=\'lower center\',ncol=4,frameon=False)\n    fig.suptitle(\'First available cycles in source order; no selection by prediction quality\\n\'+subtitle,fontsize=12)\n    fig.tight_layout(rect=(0,.10,1,.91))\n    fig.savefig(output/\'Heldout_Example_Cycles.png\',dpi=180);plt.close(fig)\n    return [output/name for name in [\'Heldout_Model_Errors.png\',\'Heldout_Waveforms.png\',\'Heldout_Example_Cycles.png\']]\n'}
for name,source in SOURCES.items():
    (CODE/name).write_text(source,encoding='utf-8')
if str(CODE) not in sys.path:
    sys.path.insert(0,str(CODE))
for module in ['data','protocol','revision_data','evaluate_saved']:
    sys.modules.pop(module,None)
spec = json.loads((STUDY/'prespecification.json').read_text())
for name in ['data.py','protocol.py','revision_data.py']:
    actual = hashlib.sha256((CODE/name).read_bytes()).hexdigest()
    if actual != spec['sources'].get(name):
        raise ValueError(f'{name}: data/preprocessing code differs from the saved training experiment')
print('Data and preprocessing source hashes match the saved experiment.')

## Verify the held-out data and evaluate available models

The evaluator checks that SUB01 is absent from every saved training-participant list, that each model belongs to the original data and selection, and that cached targets/cycle IDs match the original held-out data. Missing models are reported explicitly. Invalid artifacts are not included in comparisons.

For optional fresh inference, TensorFlow/Keras and scikit-learn/joblib must match the versions recorded by the training experiment. Cached-prediction evaluation does not load model weights and does not require these training versions. If Colab reports a version mismatch, use `cached_only` or the original training environment; the notebook will not automatically alter packages.

In [ ]:
from revision_data import legacy_cohort, coordinate_dataset
from evaluate_saved import evaluate_available, make_plots

if COORDINATES not in ('absolute','translation'):
    raise ValueError('This companion uses the supplied scalar-coordinate ZIP. Additional coordinate modes require their verified enriched bundle.')
if spec['config']['coordinate_mode'] != COORDINATES:
    raise ValueError('Coordinate mode differs from the saved experiment')
cohort = legacy_cohort(DATA_ZIP,TASK)
dataset = coordinate_dataset(cohort,COORDINATES)
if dataset.fingerprint != spec['data_fingerprint']:
    raise ValueError('Input data/preprocessing fingerprint differs from saved training data')
inventory, summary, predictions, test_data = evaluate_available(
    STUDY,dataset,OUTER_PARTICIPANT,OUTPUT,mode=EVALUATION_MODE)
display(inventory)
if summary.empty:
    print('No model could be evaluated. Read the specific reasons above. No result is claimed.')
else:
    print(f'Actual held-out results: {OUTER_PARTICIPANT}; {len(test_data.y)} cycles; ONE participant.')
    display(summary[['model','n_participants','n_cycles','mae','rmse','nrmse_percent',
                     'mean_cycle_waveform_r','undefined_nrmse_cycles','prediction_source']])
    comparisons = OUTPUT/'descriptive_baseline_comparisons.csv'
    if comparisons.exists():
        print('Descriptive improvement over the mean-training-waveform baseline; positive favors the tested model:')
        display(pd.read_csv(comparisons))
print('Saved:',OUTPUT)

In [ ]:
if not summary.empty:
    figures = make_plots(summary,predictions,test_data,OUTPUT)
    for path in figures:
        display(Image(filename=str(path)))
else:
    print('Plots require at least one valid held-out prediction set.')

## Interpretation

- MAE and RMSE are computed for each complete gait cycle and then averaged within the participant, matching the training workflow.
- Each cycle's NRMSE is its RMSE divided by its measured GRF range. `nrmse_percent` is the mean defined cycle NRMSE multiplied by 100. A zero-range target produces an undefined value, with the count reported.
- `mean_cycle_waveform_r` is descriptive. High waveform correlation alone does not demonstrate improvement over the mean-waveform or phase-only baselines.
- Waveform shading shows the measured cycle interquartile range. It is not a confidence interval. Example cycles are the first three in source order, not the best-predicted cycles.
- SUB01 remains **one participant**, regardless of the number of cycles or models. Do not use these cycle counts as an independent sample size for cohort significance tests or confidence intervals.
- These outputs are actual held-out prediction results for the completed models. They do not complete the full nested-LOSO study or select a final architecture. Do not use this outer test to change hyperparameters; selection must remain confined to inner validation.
- If a model is missing, do not substitute a tuning score or a model trained on SUB01. Conv1D-LSTM's successful tuning candidates have not yet produced a frozen outer refit in the interrupted run.
- Physical target units remain to be verified from the original source. The labels use “target units” rather than asserting unverified units.

In [ ]:
# Download the actual results, figures and prediction arrays for review.
DOWNLOAD_RESULTS = True
manifest = {
    'experiment':EXPERIMENT,'task':TASK,'coordinates':COORDINATES,
    'held_out_participant':OUTER_PARTICIPANT,'evaluation_mode':EVALUATION_MODE,
    'timestamp_utc':STAMP,'source_hashes':{
        name:hashlib.sha256(source.encode()).hexdigest() for name,source in SOURCES.items()}}
(OUTPUT/'notebook_manifest.json').write_text(json.dumps(manifest,indent=2))
archive = shutil.make_archive(str(Path('/content')/OUTPUT.name),'zip',OUTPUT)
print('Results archive:',archive)
if DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(archive)